In [1]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')

In [6]:
cols = ['engine_displacement', 'horsepower', 'vehicle_weight',
        'model_year', 'fuel_efficiency_mpg']

In [7]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')[cols]

In [8]:
df[['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']].isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [9]:
df['horsepower'].median()

np.float64(254.0)

In [10]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [11]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    return np.sqrt(se.mean())

target = 'fuel_efficiency_mpg'
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

hp_mean = df_train['horsepower'].mean()   # média só do treino

def prepare_X(d, fill_value):
    return d[features].fillna({'horsepower': fill_value}).values

y_train = df_train[target].values
y_val = df_val[target].values

for name, fill in [('with 0', 0), ('with mean', hp_mean)]:
    X_train = prepare_X(df_train, fill)
    X_val = prepare_X(df_val, fill)

    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)

    print(name, round(rmse(y_val, y_pred), 3))

with 0 2.205
with mean 2.202


In [12]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

# NAs preenchidos com 0, como o enunciado pede
X_train = prepare_X(df_train, 0)
X_val = prepare_X(df_val, 0)

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    print(r, round(rmse(y_val, y_pred), 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [13]:
def split_df(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]
    return df_train, df_val, df_test

scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    df_train, df_val, df_test = split_df(df, seed)

    X_train = prepare_X(df_train, 0)   # NAs com 0
    X_val = prepare_X(df_val, 0)
    y_train = df_train[target].values
    y_val = df_val[target].values

    w0, w = train_linear_regression(X_train, y_train)   # sem regularização
    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)
    scores.append(score)
    print(seed, round(score, 4))

std = np.std(scores)
print('std:', round(std, 3))

0 2.2392
1 2.2021
2 2.1634
3 2.2044
4 2.2019
5 2.2458
6 2.2697
7 2.1908
8 2.2166
9 2.207
std: 0.029


In [14]:
df_train, df_val, df_test = split_df(df, seed=9)


df_full_train = pd.concat([df_train, df_val])

X_full_train = prepare_X(df_full_train, 0)   # NAs com 0
X_test = prepare_X(df_test, 0)

y_full_train = df_full_train[target].values
y_test = df_test[target].values

w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
y_pred = w0 + X_test.dot(w)

print(round(rmse(y_test, y_pred), 3))

2.236
